In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np

문제 1 (단답형 주관식)


FCNN 모델은 이미지를 view(-1, 784)로 펼쳐서 1차원 벡터로 만들었습니다. 반면 CNN모델은 2D 이미지([1, 28, 28])를 그대로 입력받습니다.

1-1) FCNN(완전연결신경망)처럼 이미지를 1차원 벡터로 펼칠 때 발생하는 가장 큰 문제점(한계)은 무엇인가요?

1-2) CNN(합성곱 신경망)은 이 문제를 어떻게 해결하나요? (힌트: 필터)

1-1)
- 이미지를 1차원 벡터로 만들게 되면, 픽셀 간의 공간적 관계가 사라져 이미지의 중요한 요소인 공간 구조를 잃어버리게 된다.

1-2)
- CNN은 이미지를 2차원 형태로 그대로 입력받고, 필터(커널, Kernel)를 이미지 위에 이동시키며 국소적인 특징 추출한다. 이를 통해 특징맵(feature map)을 생성하는 방식으로 공간 정보가 유지된다.

문제 2 (실습 문제 - 코드 빈칸 채우기)

은닉층이 1개인 3계층 신경망(FCNN) 모델 Net_FCNN을 정의하는 코드입니다.

4개의 빈칸 (# TODO: ...)을 채워 모델을 완성하시오.
(입력 784 -> 은닉 128 -> ReLU -> 출력 10)

In [2]:
import torch.nn as nn

# 11차시_01_dl (FCNN) 모델 기준
class Net_FCNN(nn.Module):
    def __init__(self, n_input, n_hidden, n_output):
        super().__init__()

        # TODO: 1. 입력층(n_input)에서 은닉층(n_hidden)으로 가는 선형 계층 (self.l1)
        self.l1 = nn.Linear(n_input, n_hidden)

        # TODO: 2. 은닉층(n_hidden)에서 출력층(n_output)으로 가는 선형 계층 (self.l2)
        self.l2 = nn.Linear(n_hidden, n_output)

        # TODO: 3. ReLU 활성화 함수 (self.relu)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # TODO: 4. 입력 x를 l1, relu, l2 순서로 통과시켜 최종 logits 반환
        x = self.l1(x)
        x = self.relu(x)
        x = self.l2(x)
        return x

# --- 테스트 코드 (수정 불필요) ---
n_input = 784   # 28x28
n_hidden = 128
n_output = 10   # 0~9

net = Net_FCNN(n_input, n_hidden, n_output)
print(net)

dummy_input = torch.randn(100, 784) # 100개 배치, 784 특성
output = net(dummy_input)
print(f"\nFCNN 출력 Shape: {output.shape}") # [100, 10]

Net_FCNN(
  (l1): Linear(in_features=784, out_features=128, bias=True)
  (l2): Linear(in_features=128, out_features=10, bias=True)
  (relu): ReLU(inplace=True)
)

FCNN 출력 Shape: torch.Size([100, 10])


문제 3 (실습 문제 - 코드 빈칸 채우기)

CNN의 핵심 구성요소인 nn.Conv2d와 nn.MaxPool2d를 정의하는 코드입니다.

4개의 빈칸 (# TODO: ...)을 채우시오.

[요구사항]

conv1: 입력 채널 1개, 출력 채널(필터) 10개, 커널 크기 5x5

pool1: 커널 크기 2x2, 스트라이드(stride) 2

In [3]:
import torch.nn as nn

# 입력 채널 (흑백 이미지)
in_channels = 1
# 출력 채널 (필터 개수)
out_channels_1 = 10
# 커널 크기
kernel_size_conv = 5
kernel_size_pool = 2
stride_pool = 2

# TODO: 1. 입력 채널 1개, 출력 채널 10개, 커널 크기 5인 'conv1' 정의
conv1 = nn.Conv2d(
    in_channels = in_channels,
    out_channels = out_channels_1,
    kernel_size = kernel_size_conv
)

# TODO: 2. ReLU 활성화 함수 'relu' 정의
relu = nn.ReLU(inplace=True)

# TODO: 3. 커널 크기 2, 스트라이드 2인 'pool1' 정의
pool1 = nn.MaxPool2d(
    kernel_size = kernel_size_pool,
    stride = stride_pool
)

# --- 테스트 코드 (수정 불필요) ---
print(f"Conv2d Layer: {conv1}")
print(f"MaxPool2d Layer: {pool1}")

# (배치 100, 채널 1, 높이 28, 너비 28)의 가상 입력
dummy_image = torch.randn(100, 1, 28, 28)

# TODO: 4. conv1, relu, pool1 순서로 통과시키기
c_out = conv1(dummy_image)
r_out = relu(c_out)
p_out = pool1(r_out)

print(f"\n원본 Shape: {dummy_image.shape}")
print(f"Conv1 통과 후: {c_out.shape}")  # (100, 10, 24, 24)
print(f"MaxPool1 통과 후: {p_out.shape}") # (100, 10, 12, 12)

Conv2d Layer: Conv2d(1, 10, kernel_size=(5, 5), stride=(1, 1))
MaxPool2d Layer: MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)

원본 Shape: torch.Size([100, 1, 28, 28])
Conv1 통과 후: torch.Size([100, 10, 24, 24])
MaxPool1 통과 후: torch.Size([100, 10, 12, 12])


문제 4 (실습 문제 - 코드 빈칸 채우기)

CNN 모델의 forward 메소드에서, Conv/Pool 블록을 통과한 4D 텐서를 Linear 층에 입력하기 위해 1D 벡터로 펼치는(flatten) 과정입니다.

2개의 빈칸 (# TODO: ...)을 채워 올바른 크기로 view 하시오.

In [10]:
import torch
import torch.nn as nn

# --- 가상 CNN 모델의 일부 (수정 불필요) ---
class PartialCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # (Conv, Pool 가정...)
        # 마지막 Conv/Pool 통과 후 Shape이 [배치크기, 10, 12, 12]라고 가정

        # TODO: 1. 10*12*12 크기의 1D 벡터를 입력받는 'fc1' 정의
        self.fc1 = nn.Linear(1440, 50)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x_pool2): # x_pool2의 Shape: [100, 10, 12, 12]

        # TODO: 2. x_pool2 텐서를 fc1에 입력하기 위해 1D 벡터로 평탄화(flatten)
        # (힌트: 배치 크기(100)는 유지해야 함 -> [100, 1440])
        x_flat = x_pool2.view(x_pool2.shape[0], -1)

        # --- 이후 과정 (수정 불필요) ---
        x = self.fc1(x_flat)
        x = self.relu(x)
        x = self.fc2(x)
        return x, x_flat

# --- 테스트 코드 (수정 불필요) ---
net_cnn = PartialCNN()
# (가상 Conv/Pool 출력 생성: 배치 100, 채널 10, 높이 12, 너비 12)
dummy_pool_out = torch.randn(100, 10, 12, 12)

final_output, flat_output = net_cnn(dummy_pool_out)

print(f"모델:\n{net_cnn}")
print(f"\nFlatten 이전 Shape: {dummy_pool_out.shape}")
print(f"Flatten 이후 Shape: {flat_output.shape}")
print(f"최종 출력 Shape: {final_output.shape}")

모델:
PartialCNN(
  (fc1): Linear(in_features=1440, out_features=50, bias=True)
  (relu): ReLU(inplace=True)
  (fc2): Linear(in_features=50, out_features=10, bias=True)
)

Flatten 이전 Shape: torch.Size([100, 10, 12, 12])
Flatten 이후 Shape: torch.Size([100, 1440])
최종 출력 Shape: torch.Size([100, 10])


문제 5 (실습 문제 - 코드 작성)

torchvision을 사용하여 MNIST 훈련 데이터셋(train_set)과 검증 데이터셋(test_set)을 로드했습니다. DataLoader를 사용하여 train_loader와 test_loader를 생성하시오.

[요구사항]

- batch_size는 128로 설정합니다.
- train_loader는 데이터를 섞어야 합니다 (shuffle=True).
- test_loader는 데이터를 섞지 않아야 합니다 (shuffle=False).

In [17]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# --- 데이터셋 로드 (수정 불필요) ---
# (CNN은 1D가 아닌 2D 이미지를 입력받으므로 view(-1)가 없습니다)
transform_cnn = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform_cnn)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform_cnn)
# -----------------------------------

batch_size = 128

# TODO: 1. 훈련용 'train_loader' 생성 (shuffle=True)
train_loader = DataLoader(train_set, shuffle=True, batch_size=batch_size)

# TODO: 2. 검증용 'test_loader' 생성 (shuffle=False)
test_loader = DataLoader(test_set, shuffle=False, batch_size=batch_size)


# --- 결과 확인 (수정 불필요) ---
print(f"Train DataLoader 배치 개수: {len(train_loader)}")
print(f"Test DataLoader 배치 개수: {len(test_loader)}")

train_images, train_labels = next(iter(train_loader))
print(f"Train Batch 이미지 Shape: {train_images.shape}") # [128, 1, 28, 28]

test_images, test_labels = next(iter(test_loader))
print(f"Test Batch 이미지 Shape: {test_images.shape}")   # [128, 1, 28, 28]

Train DataLoader 배치 개수: 469
Test DataLoader 배치 개수: 79
Train Batch 이미지 Shape: torch.Size([128, 1, 28, 28])
Test Batch 이미지 Shape: torch.Size([128, 1, 28, 28])


[자율 학습]